In [1]:
import xarray as xr
import numpy as np
import pandas as pd


In [2]:
def load_region_matrix(csv_path, lat_size=360, lon_size=720):
    df = pd.read_csv(csv_path)

    region_matrix = np.empty((lat_size, lon_size), dtype=object)
    region_matrix[:] = None

    for _, r in df.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1
        region_matrix[I, J] = r["Rall"]

    return region_matrix


In [3]:
CSV_PATH = "../../../CSV/gridset/RIJ_17regions.csv"
region_matrix = load_region_matrix(CSV_PATH)

regions = sorted({
    r for r in region_matrix.flatten()
    if isinstance(r, str) and r.strip() != ""
})

regions_all = regions + ["ALL"]
print(regions_all)


['BRA', 'CAN', 'CHN', 'CIS', 'IND', 'JPN', 'TUR', 'USA', 'XAF', 'XE25', 'XER', 'XLM', 'XME', 'XNF', 'XOC', 'XSA', 'XSE', 'ALL']


In [4]:
def split_da_by_region(da, region_matrix, regions):
    """
    da: (time, lat, lon)
    return: (time, region, lat, lon)
    """

    out = []

    for reg in regions:
        mask = (region_matrix == reg)
        mask_da = xr.DataArray(
            mask,
            dims=("lat", "lon"),
            coords={"lat": da.lat, "lon": da.lon}
        )

        da_reg = da.where(mask_da)
        out.append(da_reg)

    # --- ALL (global) ---
    out.append(da)

    da_out = xr.concat(
        out,
        dim=pd.Index(regions + ["ALL"], name="region")
    )

    return da_out


In [5]:
NC_IN = "../../../NC/compare_diff.nc"
NC_OUT_REGION = "../../../NC/region_agri_diff_by17regions.nc"
NC_OUT_BASIN  = "../../../NC/basin_agri_diff_by17regions.nc"

ds = xr.open_dataset(NC_IN)

# --- region mode ---
da_region_split = split_da_by_region(
    ds["region_agri_diff"],
    region_matrix,
    regions
)

xr.Dataset(
    {"agri_diff": da_region_split}
).to_netcdf(NC_OUT_REGION)

print(f"[Saved] {NC_OUT_REGION}")

# --- basin mode ---
da_basin_split = split_da_by_region(
    ds["basin_agri_diff"],
    region_matrix,
    regions
)

xr.Dataset(
    {"agri_diff": da_basin_split}
).to_netcdf(NC_OUT_BASIN)

print(f"[Saved] {NC_OUT_BASIN}")


[Saved] ../../../NC/region_agri_diff_by17regions.nc
[Saved] ../../../NC/basin_agri_diff_by17regions.nc


In [6]:
from libpysal.weights import lat2W
from libpysal.weights.util import w_subset
from esda import Moran
import warnings
warnings.filterwarnings("ignore", message=".*is an island.*")


In [ ]:
def moran_from_split_nc(
    da,
    years,
    lat_dim="lat",
    lon_dim="lon"
):
    """
    da: (time, region, lat, lon)
    """

    nlat = da.sizes[lat_dim]
    nlon = da.sizes[lon_dim]

    print(f"[Init] building spatial weights ({nlat} x {nlon}) ...")
    w_full = lat2W(nlat, nlon)
    w_full.transform = "r"
    print("[Init] spatial weights ready\n")

    regions = list(da.region.values)
    n_years = len(years)
    n_regions = len(regions)

    results = []
    t0_all = time.time()

    for ti, year in enumerate(years):
        print(f"\n[Year {year}] ({ti+1}/{n_years})")
        t0_year = time.time()

        for ri, reg in enumerate(regions):
            print(f"  → region {reg} ({ri+1}/{n_regions})", end=" ... ")
            sys.stdout.flush()

            arr = da.sel(region=reg).isel(time=ti).values
            arr = np.nan_to_num(arr, nan=0.0)

            x = arr.flatten()
            mask = ~np.isnan(arr.flatten())

            if mask.sum() < 10:
                print("skipped (too few cells)")
                continue

            w_reg = w_subset(w_full, np.where(mask)[0])
            w_reg.transform = "r"

            mi = Moran(x[mask], w_reg, permutations=0)

            results.append({
                "year": year,
                "region": reg,
                "moran_I": mi.I
            })

            print(f"I = {mi.I:.4f}")

        dt_year = time.time() - t0_year
        print(f"[Year {year} done] elapsed: {dt_year/60:.1f} min")

    dt_all = time.time() - t0_all
    print(f"\n[ALL DONE] total time: {dt_all/60:.1f} min")

    return pd.DataFrame(results)


In [ ]:
years = list(range(2010, 2101, 10))

# --- region mode ---
ds_r = xr.open_dataset(NC_OUT_REGION)
df_region = moran_from_split_nc(ds_r["agri_diff"], years)
df_region["mode"] = "region"

# --- basin mode ---
ds_b = xr.open_dataset(NC_OUT_BASIN)
df_basin = moran_from_split_nc(ds_b["agri_diff"], years)
df_basin["mode"] = "basin"

df_all = pd.concat([df_region, df_basin], ignore_index=True)
df_all["scope"] = np.where(df_all["region"] == "ALL", "global", "regional")
df_all = df_all[["year", "scope", "region", "mode", "moran_I"]]
df_all["moran_I"] = df_all["moran_I"].round(3)

OUT_CSV = "../../../CSV/moran_interannual_diff_global_and_regional_from_splitNC.csv"
df_all.to_csv(OUT_CSV, index=False)

print(f"[Saved] {OUT_CSV}")
